# 02 · Distribution feeder with rooftop solar (3B): CIGRE MV benchmark

**Question for this notebook:** how much rooftop solar can a medium-voltage feeder take before voltages rise too far or cables overload, and what happens when individual feeder sections fail?

| Step | Method | Main reference |
|---|---|---|
| PV output | NASA POWER hourly irradiance, temperature and wind → pvlib (Erbs decomposition, Hay–Davies transposition, SAPM cell temperature, PVWatts) | Holmgren et al. (2018) |
| Feeder | CIGRE MV benchmark, 20 kV, two feeders with open tie switches | CIGRE TB 575 (Strunz et al., 2014) |
| Hosting capacity | PV penetration sweep at high sun and light load: voltage rise, cable loading, reverse flow | Walling et al. (2008); Ismael et al. (2019) |
| Mitigation | inverter reactive power (fixed power factor 0.95 absorbing) | Tonkoski et al. (2011); IEEE 1547-2018 |
| Section outages | each cable/line section out → unsupplied load → restore by closing tie switches | — |

**Outputs** (in `results/02_feeder/`): `pv_profile_per_kwp.csv` (used again by notebook 05), `hosting_capacity.csv`, `daily_timeseries.csv`, `feeder_outages.csv`, figures.

**Data:** NASA POWER hourly API (free, no key). The files are cached in the project folder, and notebook 03 reuses them.

## 0. Setup

In [ ]:
!pip install -q pandapower pvlib

In [ ]:
import os, sys, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})

# ---- Shared project settings (identical in all five notebooks) ----
PROJECT_ID = "goggle-earth-499404"          # Google Cloud project registered for Earth Engine
SITE = "Nagpur, Maharashtra"
LAT, LON, ELEV_M = 21.15, 79.09, 310
TZ = "Asia/Kolkata"

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True                             # keep results in Google Drive so the notebooks can share them
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/grid_stress_nagpur"
else:
    ROOT = os.path.abspath("grid_stress_nagpur")

CACHE = os.path.join(ROOT, "cache")
RES = {k: os.path.join(ROOT, "results", k) for k in
       ["01_contingency", "02_feeder", "03_heat", "04_cascade", "05_mitigation"]}
for d in [CACHE, *RES.values()]:
    os.makedirs(d, exist_ok=True)
OUT = RES["02_feeder"]

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")
    plt.show()

def require(path, notebook):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run notebook {notebook} first (with USE_DRIVE = True).")
    return path

print("Project folder:", ROOT)

In [ ]:
# ---- NASA POWER hourly data (shared by notebooks 02, 03 and 05; cached in the project folder) ----
import requests

POWER_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"
POWER_PARAMS = ["ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]

def fetch_power_year(year, lat=LAT, lon=LON, retries=3):
    """One year of hourly GHI (W/m²), 2-m air temperature (°C) and 10-m wind (m/s), in UTC."""
    f = os.path.join(CACHE, f"power_utc_{lat:.2f}_{lon:.2f}_{year}.csv")
    if os.path.exists(f):
        return pd.read_csv(f, index_col=0, parse_dates=True)
    params = {"parameters": ",".join(POWER_PARAMS), "community": "RE",
              "longitude": lon, "latitude": lat, "start": f"{year}0101", "end": f"{year}1231",
              "format": "JSON", "time-standard": "UTC"}
    for k in range(retries):
        try:
            r = requests.get(POWER_URL, params=params, timeout=180)
            r.raise_for_status()
            break
        except Exception:
            if k == retries - 1:
                raise
            time.sleep(10 * (k + 1))
    p = r.json()["properties"]["parameter"]
    df = pd.DataFrame({k: pd.Series(v) for k, v in p.items()})
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    df = df.replace(-999.0, np.nan).sort_index()
    df.columns = [{"ALLSKY_SFC_SW_DWN": "ghi", "T2M": "t2m", "WS10M": "ws10m"}[c] for c in df.columns]
    df.to_csv(f)
    return df

def load_power(years):
    return pd.concat([fetch_power_year(y) for y in years])

## 1. Parameters
- **Load shapes.** CIGRE gives peak loads only. The 24-hour shapes below are *synthetic*: residential load peaks in the evening (cooling and lighting), commercial/industrial load peaks during working hours. They are typical of Indian summer urban feeders; replace them with MSEDCL feeder data if you can obtain it.
- **Voltage limits.** ±5% for normal operation (IEEE 1547 / ANSI C84.1 range A). Check against the CEA Grid Standards and the Maharashtra supply code before finalising. Emergency limits apply after a contingency.
- **PV placement.** PV goes on the distribution buses (3–11, 13, 14). Buses 1 and 12 carry the aggregated loads of the rest of the network in the CIGRE model, so no PV is added there.

In [ ]:
POWER_YEARS = range(2005, 2025)       # NASA POWER hourly starts in 2001
SEASON = (3, 6)                       # March–June

PV_TILT, PV_AZIMUTH = 20.0, 180.0     # roughly latitude tilt, facing south
PV_GAMMA = -0.0037                    # power temperature coefficient, 1/°C (mono-Si)
PV_LOSSES = 0.14                      # soiling, wiring, mismatch, availability
DC_AC_RATIO = 1.1

# Synthetic hourly load shapes (fraction of peak), index = hour of day (IST, centre of the hour)
RES_SHAPE = np.array([0.55, 0.50, 0.48, 0.46, 0.46, 0.50, 0.58, 0.62, 0.60, 0.55, 0.52, 0.52,
                      0.55, 0.58, 0.60, 0.62, 0.65, 0.72, 0.85, 0.95, 1.00, 0.95, 0.82, 0.68])
CI_SHAPE = np.array([0.35, 0.33, 0.32, 0.32, 0.33, 0.38, 0.50, 0.65, 0.80, 0.92, 0.98, 1.00,
                     0.97, 0.98, 0.97, 0.93, 0.85, 0.72, 0.60, 0.52, 0.47, 0.43, 0.40, 0.37])
LIGHT_LOAD = 0.70                     # holiday / mild-weather day relative to the summer shape

V_MIN, V_MAX = 0.95, 1.05             # normal operating limits, p.u.
V_MIN_EMERG, V_MAX_EMERG = 0.90, 1.10 # emergency limits after a contingency
LOADING_MAX, LOADING_MAX_EMERG = 100.0, 120.0

PV_BUSES = [3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14]
PENETRATIONS = np.round(np.arange(0, 4.01, 0.1), 2)   # PV kWp / bus peak load (kW)
PEN_TS = 1.5                          # penetration used in the daily time-series runs
SWITCHING_H = 1.0                     # manual switching time to restore via tie switches, h
REPAIR_H_FEEDER = 6.0                 # cable/line section repair time, h

## 2. Rooftop PV output for Nagpur from NASA POWER + pvlib
The model chain, step by step:
1. **Sun position** at the middle of each hour (NASA POWER values are hourly means).
2. **Decomposition** of global horizontal irradiance (GHI) into direct normal (DNI) and diffuse (DHI) using the Erbs correlation.
3. **Transposition** to the tilted module plane (plane-of-array irradiance, POA) with the Hay–Davies sky model.
4. **Cell temperature** from POA, air temperature and wind speed (Sandia SAPM model, roof-mounted glass/glass modules). Nagpur's hot afternoons cost several percent of output.
5. **DC power** with the PVWatts model: \(P_{dc} = \frac{POA}{1000}\,P_{dc0}\,[1 + \gamma (T_{cell} - 25)]\), then system losses.
6. **AC power** with the PVWatts inverter model (nominal efficiency 96%, DC/AC ratio 1.1).

In [ ]:
power = load_power(POWER_YEARS)
print(power.shape, power.index.min(), "→", power.index.max(), "(UTC)")
power.describe().T

In [ ]:
# ---- PV output per kWp from NASA POWER with pvlib (notebook 02; profile reused in 05) ----
import pvlib


def pv_output_per_kwp(power, tilt=20.0, azimuth=180.0, gamma=-0.0037, losses=0.14,
                      dc_ac_ratio=1.1, eta_inv=0.96, mount="close_mount_glass_glass"):
    """Hourly AC output (kW per kWp DC) of a rooftop PV system.

    Chain: solar position -> Erbs GHI decomposition (DNI, DHI) -> Hay-Davies transposition to the
    tilted plane -> SAPM cell temperature -> PVWatts DC model -> losses -> PVWatts inverter."""
    times = pd.DatetimeIndex(power.index).tz_localize("UTC") + pd.Timedelta("30min")   # mid-hour sun position
    loc = pvlib.location.Location(LAT, LON, altitude=ELEV_M)
    sp = loc.get_solarposition(times)
    ghi = power["ghi"].clip(lower=0).values
    dec = pvlib.irradiance.erbs(ghi, sp["zenith"].values, times)
    poa = pvlib.irradiance.get_total_irradiance(
        surface_tilt=tilt, surface_azimuth=azimuth,
        solar_zenith=sp["apparent_zenith"].values, solar_azimuth=sp["azimuth"].values,
        dni=np.nan_to_num(dec["dni"]), ghi=ghi, dhi=np.nan_to_num(dec["dhi"]),
        dni_extra=pvlib.irradiance.get_extra_radiation(times).values, model="haydavies")
    poa_g = np.nan_to_num(np.asarray(poa["poa_global"], float))
    prm = pvlib.temperature.TEMPERATURE_MODEL_PARAMETERS["sapm"][mount]
    tcell = pvlib.temperature.sapm_cell(poa_g, power["t2m"].values, power["ws10m"].values, **prm)
    pdc = pvlib.pvsystem.pvwatts_dc(poa_g, tcell, 1.0, gamma) * (1 - losses)
    pac = pvlib.inverter.pvwatts(pdc, 1.0 / dc_ac_ratio, eta_inv_nom=eta_inv)
    out = pd.DataFrame({"pv_kw_per_kwp": np.clip(np.nan_to_num(pac), 0, None),
                        "poa": poa_g, "tcell": tcell}, index=power.index)
    return out

In [ ]:
pv = pv_output_per_kwp(power, PV_TILT, PV_AZIMUTH, PV_GAMMA, PV_LOSSES, DC_AC_RATIO)
pv["ist"] = pv.index + pd.Timedelta(hours=5, minutes=30)          # start of the averaging hour, IST
pv["ist_hour"] = (pv.index + pd.Timedelta(hours=6)).hour            # centre of the averaging hour, IST
pv["ghi"], pv["t2m"], pv["ws10m"] = power["ghi"], power["t2m"], power["ws10m"]
pv.to_csv(os.path.join(OUT, "pv_profile_per_kwp.csv"), index_label="time_utc")

season = pv[(pv.ist.dt.month >= SEASON[0]) & (pv.ist.dt.month <= SEASON[1])]
annual_yield = pv.groupby(pv.ist.dt.year)["pv_kw_per_kwp"].sum()
annual_yield = annual_yield[annual_yield.index.isin(list(POWER_YEARS))]        # full years only
P_DESIGN = season["pv_kw_per_kwp"].quantile(0.99)
print(f"Specific yield: {annual_yield.mean():.0f} kWh/kWp/yr (range {annual_yield.min():.0f}–{annual_yield.max():.0f})")
print(f"Capacity factor: {annual_yield.mean() / 8760:.1%}")
print(f"Design PV output for hosting capacity (99th pct, Mar–Jun): {P_DESIGN:.3f} kW per kWp")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
prof = season.groupby("ist_hour")["pv_kw_per_kwp"].agg(["mean", lambda x: x.quantile(0.9), lambda x: x.quantile(0.1)])
ax[0].fill_between(prof.index, prof.iloc[:, 2], prof.iloc[:, 1], alpha=0.3, color="#e0a100", label="10–90th pct")
ax[0].plot(prof.index, prof["mean"], color="#b07900", label="mean"); ax[0].legend(frameon=False)
ax[0].set_xlabel("Hour (IST)"); ax[0].set_ylabel("kW per kWp"); ax[0].set_title("PV output, March–June")
mon = pv.groupby([pv.ist.dt.year, pv.ist.dt.month])["pv_kw_per_kwp"].sum().groupby(level=1).mean()
ax[1].bar(mon.index, mon.values, color="#e0a100"); ax[1].set_xlabel("Month"); ax[1].set_ylabel("kWh per kWp")
ax[1].set_title("Monthly yield (monsoon dip)")
ax[2].scatter(season.poa[season.poa > 50], season.tcell[season.poa > 50], s=1, alpha=0.1, color="#c0392b")
ax[2].set_xlabel("POA irradiance (W/m²)"); ax[2].set_ylabel("Cell temperature (°C)"); ax[2].set_title("Module heating")
savefig("fig01_pv_profile.png")

## 3. CIGRE medium-voltage feeder
The network has two 20 kV feeders supplied by two 25 MVA transformers from a 110 kV grid. Three tie switches (S1, S2, S3) are normally open, so the feeders run radially.

In [ ]:
import pandapower as pp
import pandapower.networks as pn
import pandapower.topology as top

def make_feeder():
    return pn.create_cigre_network_mv(with_der=False)

net0 = make_feeder()
IS_RES = net0.load.name.str.contains(" R")
PEAK_BUS = net0.load.groupby("bus").p_mw.sum()
pp.runpp(net0)
print("Peak-load base case: Vmin = %.3f p.u., max cable loading = %.0f%%, transformer loading = %s%%"
      % (net0.res_bus.vm_pu.min(), net0.res_line.loading_percent.max(), net0.res_trafo.loading_percent.round(0).tolist()))
print("Tie switches:", net0.switch[~net0.switch.closed][["name", "bus", "element"]].to_dict("records"))
pd.DataFrame({"peak load (MW)": PEAK_BUS, "PV bus": PEAK_BUS.index.isin(PV_BUSES)})

In [ ]:
def set_operating_point(net, hour, load_factor=1.0):
    # Scale CIGRE peak loads to a given IST hour using the synthetic shapes
    net.load.loc[IS_RES, "scaling"] = RES_SHAPE[hour] * load_factor
    net.load.loc[~IS_RES, "scaling"] = CI_SHAPE[hour] * load_factor

def add_pv(net, penetration, pv_per_kwp, cos_phi=1.0):
    # PV at each PV bus: installed kWp = penetration x bus peak load; output = kWp x pv_per_kwp
    for b in PV_BUSES:
        p = penetration * PEAK_BUS[b] * pv_per_kwp
        q = -p * np.tan(np.arccos(cos_phi))           # negative = absorbing reactive power
        pp.create_sgen(net, b, p_mw=p, q_mvar=q, type="PV", name=f"PV {b}")

MV_BUSES = net0.bus.index[net0.bus.vn_kv < 100]      # 20 kV buses only (the 110 kV slack is fixed at 1.03 p.u.)

def feeder_metrics(net):
    vm = net.res_bus.vm_pu.loc[MV_BUSES]
    return dict(vmax=vm.max(), vmin=vm.min(),
                max_line_loading=net.res_line.loading_percent.max(),
                trafo_p_mw=net.res_trafo.p_hv_mw.sum(),
                reverse_flow=bool((net.res_trafo.p_hv_mw < 0).any()))

## 4. Hosting capacity: voltage rise and overloads at high sun
**Worst case for voltage rise:** high PV output (99th percentile of March–June) at solar noon together with a light midday load. PV injects power into the feeder; the current flowing back through the line resistance raises voltage along the feeder: \(\Delta V \approx (R\,P - X\,Q)/V\).

We sweep the penetration (installed PV as a multiple of each bus's peak load) at unity power factor and with inverters absorbing reactive power at PF 0.95.

In [ ]:
NOON = 12
rows = []
for cos_phi in [1.0, 0.95]:
    for pen in PENETRATIONS:
        n = make_feeder()
        set_operating_point(n, NOON, LIGHT_LOAD)
        add_pv(n, pen, P_DESIGN, cos_phi)
        pp.runpp(n)
        rows.append(dict(cos_phi=cos_phi, penetration=pen,
                         pv_mw=sum(pen * PEAK_BUS[b] for b in PV_BUSES), **feeder_metrics(n)))
hc = pd.DataFrame(rows)
hc.to_csv(os.path.join(OUT, "hosting_capacity.csv"), index=False)

def first_violation(d, col, lim, above=True):
    bad = d[d[col] > lim] if above else d[d[col] < lim]
    return bad.penetration.min() if len(bad) else np.nan

hc_summary = []
for cos_phi, d in hc.groupby("cos_phi"):
    pv_v = first_violation(d, "vmax", V_MAX); pv_l = first_violation(d, "max_line_loading", LOADING_MAX)
    lim = np.nanmin([pv_v, pv_l]) - 0.1
    hc_summary.append(dict(cos_phi=cos_phi, voltage_limit_at=pv_v, thermal_limit_at=pv_l,
                           reverse_flow_from=d[d.reverse_flow].penetration.min(),
                           hosting_capacity_penetration=lim,
                           hosting_capacity_mw=sum(lim * PEAK_BUS[b] for b in PV_BUSES)))
hc_summary = pd.DataFrame(hc_summary).set_index("cos_phi")
hc_summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for cos_phi, d in hc.groupby("cos_phi"):
    lab = "unity PF" if cos_phi == 1 else f"PF {cos_phi} absorbing"
    ax[0].plot(d.penetration, d.vmax, label=lab)
    ax[1].plot(d.penetration, d.max_line_loading, label=lab)
ax[0].axhline(V_MAX, color="#c0392b", ls="--", lw=1); ax[0].set_ylabel("Highest bus voltage (p.u.)")
ax[1].axhline(LOADING_MAX, color="#c0392b", ls="--", lw=1); ax[1].set_ylabel("Highest cable/line loading (%)")
for a in ax:
    a.set_xlabel("PV penetration (kWp / peak load)"); a.legend(frameon=False)
fig.suptitle(f"Hosting capacity, CIGRE MV feeder, noon, PV at {P_DESIGN:.2f} kW/kWp, load at {LIGHT_LOAD:.0%} of summer shape", fontsize=10)
savefig("fig02_hosting_capacity.png")

## 5. A day on the feeder: clear hot day vs hazy day
Hourly power flows for the sunniest and the dullest March–June day in the record, at penetration `PEN_TS`, with the summer load shape.

In [ ]:
daily = season.groupby(season.ist.dt.date)["pv_kw_per_kwp"].sum()
days = {"sunniest day": daily.idxmax(), "dullest day": daily.idxmin()}
ts_rows = []
for label, d in days.items():
    day = season[season.ist.dt.date == d].set_index("ist_hour")["pv_kw_per_kwp"].reindex(range(24), fill_value=0)
    for h in range(24):
        n = make_feeder()
        set_operating_point(n, h, 1.0)
        add_pv(n, PEN_TS, day[h])
        pp.runpp(n)
        ts_rows.append(dict(day=label, date=str(d), hour=h, pv_kw_per_kwp=day[h], **feeder_metrics(n)))
ts = pd.DataFrame(ts_rows)
ts.to_csv(os.path.join(OUT, "daily_timeseries.csv"), index=False)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for label, d in ts.groupby("day"):
    ax[0].plot(d.hour, d.trafo_p_mw, label=f"{label} ({d.date.iloc[0]})")
    ax[1].plot(d.hour, d.vmax, label=label); ax[1].plot(d.hour, d.vmin, ls=":", color=ax[1].lines[-1].get_color())
    ax[2].plot(d.hour, d.max_line_loading, label=label)
ax[0].set_ylabel("Power from 110 kV grid (MW)"); ax[0].legend(frameon=False, fontsize=7)
ax[1].axhline(V_MAX, color="#c0392b", ls="--", lw=1); ax[1].axhline(V_MIN, color="#c0392b", ls="--", lw=1)
ax[1].set_ylabel("Vmax (solid), Vmin (dotted), p.u.")
ax[2].axhline(LOADING_MAX, color="#c0392b", ls="--", lw=1); ax[2].set_ylabel("Highest loading (%)")
for a in ax: a.set_xlabel("Hour (IST)")
fig.suptitle(f"Daily operation at PV penetration {PEN_TS}", fontsize=10)
savefig("fig03_daily_operation.png")

Midday PV pushes voltages up while the evening residential peak, after sunset, pulls them down. Rooftop PV therefore does not relieve the evening peak; that needs storage (notebook 05).

## 6. Feeder-section outages and restoration by tie switches
Each cable or overhead section is taken out in turn. We record the load cut off from the substation, then search all combinations of the three tie switches for the one that restores the most load within *emergency* limits. Restored load is back after `SWITCHING_H`; the rest waits for repair (`REPAIR_H_FEEDER`).

Rooftop PV cannot keep an isolated section alive: grid-tied inverters must disconnect within seconds when the grid is lost (anti-islanding, IEEE 1547 / CEA connectivity rules). PV with a battery and an islanding inverter changes this, which is the point of notebook 05.

In [ ]:
import itertools, copy

def check(net):
    try:
        pp.runpp(net)
    except pp.LoadflowNotConverged:
        return False
    vm = net.res_bus.vm_pu.loc[MV_BUSES].dropna()
    return (vm.min() >= V_MIN_EMERG) and (vm.max() <= V_MAX_EMERG) and (net.res_line.loading_percent.max() <= LOADING_MAX_EMERG)

def unsupplied_load(net):
    unsup = top.unsupplied_buses(net)
    return float((net.load.p_mw * net.load.scaling)[net.load.bus.isin(unsup)].sum()), sorted(unsup)

ties = net0.switch.index[~net0.switch.closed].tolist()
out_rows = []
for hour_label, hour in [("evening peak 20:00", 20), ("noon 12:00", 12)]:
    for li in net0.line.index:
        n = make_feeder(); set_operating_point(n, hour, 1.0)
        n.line.at[li, "in_service"] = False
        lost, buses = unsupplied_load(n)
        if lost == 0:
            continue
        best = (lost, ())
        for k in range(1, len(ties) + 1):
            for combo in itertools.combinations(ties, k):
                m = copy.deepcopy(n)
                m.switch.loc[list(combo), "closed"] = True
                rem, _ = unsupplied_load(m)
                if rem < best[0] - 1e-6 and check(m):
                    best = (rem, combo)
        rem, combo = best
        out_rows.append(dict(time=hour_label, section=f"L{li} ({net0.line.from_bus[li]}-{net0.line.to_bus[li]})",
                             buses_cut=len(buses), load_cut_mw=lost, restored_mw=lost - rem, remaining_mw=rem,
                             tie_switches_closed=",".join(net0.switch.name[list(combo)].fillna("tie").astype(str)) if combo else "none",
                             unserved_mwh=(lost * SWITCHING_H + rem * (REPAIR_H_FEEDER - SWITCHING_H))))
fo = pd.DataFrame(out_rows)
fo.to_csv(os.path.join(OUT, "feeder_outages.csv"), index=False)
fo

In [ ]:
d = fo[fo.time.str.startswith("evening")].sort_values("load_cut_mw")
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(d.section, d.restored_mw, color="#4a9c5d", label="restored by tie switches")
ax.barh(d.section, d.remaining_mw, left=d.restored_mw, color="#c0392b", label="waits for repair")
ax.set_xlabel("Load cut off (MW), evening peak"); ax.legend(frameon=False)
ax.set_title("Feeder-section outages, CIGRE MV")
savefig("fig04_feeder_outages.png")

## 7. Notes for the technical note
- **PV.** NASA POWER hourly data (2005–2024) run through the pvlib chain gives the specific yield and the design output. The same per-kWp profile is reused for battery sizing in notebook 05.
- **Hosting capacity.** It is limited by voltage rise first, and reactive-power absorption raises it. Report the penetration and MW at each limit.
- **Limitations:**
  - European benchmark feeder.
  - Synthetic load shapes.
  - Balanced three-phase model (real LV rooftop PV is unbalanced, so local voltage rise is worse).
  - No on-load tap changer control.
  - Restoration only by the three CIGRE tie switches.

**Next:** notebook 03 (heat derating, Earth Engine).